$$
\def\vec#1{\overrightarrow{#1}} 
\def\dint{\displaystyle \int}%
\def\dsum{\displaystyle \sum}%
\def\dsqrt{\displaystyle \sqrt}%
\def\df#1#2{{\dfrac{d #1}{d #2}}}
\def\tens#1{\underline{\underline{#1}}}
\def\vect#1{\left\{\mathbf{#1}\right\}}
\def\mat#1{\left[\mathbf{#1}\right]}
\def\vecd#1#2{\begin{Bmatrix} #1 \\ #2 \end{Bmatrix}}
$$

# EF d'Hermite en Statique Linéaire 

**Formulation et mise en oeuvre d'éléments finis de poutres à interpolations d'Hermite en flexion et linéaire en tension**

On se réfère ici aux détails donnés dans l'introduction et au Chapitre 6 de https://www.emse.fr/~drapier/index_fichiers/CoursPDF/Meca-Structu2A/SDrapier_Meca-struct-num_2026.pdf

***S. Drapier, Novembre 2024***

## Discrétisation de la formulation en statique linéaire

Pour le *problème statique en HPP*, les problèmes de tension et flexion sont totalement découplés : on superpose les contributions dans le PPV pour former le problème à résoudre. On rappelle que la formulation par élément fini repose sur l'approximation de **Galerkin**, dans laquelle *l'approximation de l'inconnue et de la fonction test sont prises dans la même base de fonctions; la fonction test étant C.A.(0) tandis que la fonction inconnue est C.A. par construction*.

**Remarque générale :** on raisonnera souvent sur la forme intégrale faible limitée spatialement à un seul élément. On notera que cette forme faible n'est pas nulle car chaque élémement est en équilibre *aussi* sous des *conditions de raccord*, *i.e.* l'effet des contributions de ses voisins et potentiellement celui des conditions aux limites appliquées au problème global. Ces conditions seront prises en compte plus tard, après assemblage des contributions élémentaires comme on le pratique dans l'approximation de Galerkin.


### Tension-compression
Pour un seul élément, la forme à discrétiser en tension seul s'écrit assez simplement :
<a id="eq2.1"></a>
$$\begin{equation}
	\begin{array}{l}
		\delta \mathcal{P}_{int}(\delta u(x)) + \delta \mathcal{P}_{ext}(\delta u(x))
		=0, \ \forall \delta u(x)\:C.A.(0)  \xrightarrow{\text{élémentaire}}
		-\dint_{x_1}^{x_2} {N^h(x)\:\delta u'^h(x) \:dx^h +\dint_{x_1}^{x_2}  p_x(x)\:\delta u^h(x)}\:dx^h 
	\end{array}
\tag{2.1}
\end{equation}$$
En introduisant la discrétisation linéaire (<a href="main_EF_Hermite.ipynb#eq1.7">Eq. (1.7)</a>) pour le champ recherché et le champ test sous la forme $u^{h,e}(x)=\dsum_{i=1}^2 
N_i(x)u_i=\dsum_{i=1}^2 N_i(x) \ q^e_{i,t}$ et *en considérant des propriétés constantes par élément*, on obtient le système algébrique élémentaire <a id="eq2.2"></a>
$$\begin{equation}
	\begin{array}{l}
	\dsum_{i=1}^2  \left\{  \dsum_{j=1}^2  \dint_{x_1}^{x_2} \left(- E^eS^e\delta q_{i,t}^{e} \dfrac{d{N}_i}{dx} \dfrac{d{N}_j}{dx^2} q_{j,t}^{e} + p_y \: \delta q_{i,t}^{e} \mathcal{N}_i  \right)\:dx^h  \right\} \\[1.5em]
	\Leftrightarrow	\dsum_{i=1}^2  \delta q_{i,t}^{e} \left\{  \dsum_{j=1}^2   \left( -E^{e}S^{e} \dint_{x_1}^{x_2} \dfrac{d{N}_i}{dx} \dfrac{d{N}_j}{dx} \:dx^h \right) q_{j,t}^{e} + p_y^{e}  \dint_{x_1}^{x_2} \mathcal{N}_i  \:dx^h  \right\}\\[1.5em]
	\Leftrightarrow \dsum_{i=1}^2  \dsum_{j=1}^2 \delta q^{e}_i K^{e}_{ij} \; q_j^{e} - \dsum_{i=1}^2 \delta q^{e}_i f_i^{e} 
\end{array}
\tag{2.2}
\end{equation}$$
avec la matrice de rigidité symétrique définie positive et le second membre, définis dans le repère local de la poutre
<a id="eq2.3"></a>
$$\begin{equation}
	\begin{array}{rcl}
		\text{Matrice de rigidit\'{e} élémentaire en tension}\hspace{15pt}
		K_{ij,t}^e&=&E^eS^e \dint_{x_1}^{x_2} \dfrac{dN_j(x)}{dx} \dfrac{dN_i(x)}{dx} \;dx^h \\[+1.5em]
		%&=&\left\{ \begin{array}{rcl} \dfrac{E^eS^e}{l_e} \text{ si }i=j \\[+0.5em] -\dfrac{E^eS^e}{l_e} \text{ si } i\neq
		%	j\end{array}\right.\\[+2em]
		\text{Vecteur des efforts ext\'{e}rieurs répartis selon } \vec x \hspace{15pt}
		f_{i, px}^e&=&p_x \dint_{x_1}^{x_2} \;N_i(x)\;dx^h = p_x \left(\dfrac{l_e}{2}\right)
	\end{array}
\tag{2.3}
\end{equation}$$

On aurait pu directement calculer ces grandeurs élémentaires à partir des approximations des déformations (<a href="main_EF_Hermite.ipynb#eq1.15">Eq. (1.15)</a>), car $N^h(x) = E^e S^e e_x^h(x)=\mat{\mathcal L_t}  \mat{B_L^e}_t \vect{u^e_t}$. L'expression générale de la raideur et des efforts répartis est donc :
<a id="eq2.4"></a>
$$\begin{equation} 
\begin{array}{ccc}
	\mat{K^e_t} &=& \dint_e \mat{B_L^e}^T_t  \mat{\mathcal L_t}  \mat{B_L^e}_t  dx^h &&
	\vect{f^e_{px} }&= &\dint_e p_x <N(x^h)>^T  dx^h
\end{array}
\tag{2.4}
\end{equation}$$
avec $\mat{\mathcal L_t}$ la sous-matrice de comportement en membrane, limiteé ici à la seule rigidité $E^eS^e$. Tous calculs faits, ces expressions ont la forme suivante dans le repère local de la poutre :
<a id="eq2.5"></a>
$$\begin{equation}
		\begin{array}{lr}
		\mat{K^e_t}= \dfrac{E^e S^e}{l_e}\begin{bmatrix}
			1 & -1 \\
			-1 & 1 \end{bmatrix} & \ \ \
		\vect{ f^e_{px}} = p_x\left(\dfrac{l_e}{2} \right)	\begin{Bmatrix}	1  \\  1 \end{Bmatrix}
	\end{array}
\tag{2.5}
\end{equation}$$

### Flexion
La même démarche pour la flexion part de la forme intégrale faible élémentaire
<a id="eq2.6"></a>
$$-\dint_{x_1}^{x_2} {M^h(x)\:\delta v''^h(x) \:dx^h +\dint_{x_1}^{x_2}  p_y(x)\:\delta v^h(x)}\:dx^h
\tag{2.6}
$$
qui est discrétisée avec l'approximation nodale $v^{h,e}(x)=\dsum_{i=1}^4 \mathcal N_i(x)\ q^e_{i,f}$ (<a href="main_EF_Hermite.ipynb#eq1.9">Eq. (1.9)</a>)
<a id="eq2.7"></a>
$$\begin{equation}
	\begin{array}{l}
	\dsum_{i=1}^4  \left\{  \dsum_{j=1}^4  \dint_{x_1}^{x_2} \left(- E^eI^e\delta q_{i,f}^{e} \dfrac{d^2\mathcal{N}_i}{dx^2} \dfrac{d^2\mathcal{N}_j}{dx^2} q_{j,t}^{e} + p_y \: \delta q_i^{e} \mathcal{N}_i  \right)\:dx^h  \right\} \\[1.5em]
	\Leftrightarrow	\dsum_{i=1}^4  \delta q_{i,f}^{e} \left\{  \dsum_{j=1}^4   \left( -E^{e}I^{e} \dint_{x_1}^{x_2} \dfrac{d^2\mathcal{N}_i}{dx^2} \dfrac{d^2\mathcal{N}_j}{dx^2}  \:dx^h \right) q_{j,f}^{e} + p_y^{e}  \dint_{x_1}^{x_2} \mathcal{N}_i  \:dx^h  \right\}
\end{array}
\tag{2.7}
\end{equation}
$$
ce qu'on met, comme pour la tension, sous la forme générique $ \delta q^{e}_{i,f} \left(K^{e}_{ij} \; q_{j,f}^{e} - f_{i,f}^{e}\right)$, 
<a id="eq2.8"></a>
$$\begin{equation}
	\begin{array}{rcl}
		\text{Matrice de rigidit\'{e} élémentaire en flexion}\hspace{15pt}
		K_{ij,f}^e&=&E^{e}I^{e} \dint_{x_1}^{x_2} \dfrac{d^2\mathcal{N}_i}{dx^2} \dfrac{d^2\mathcal{N}_j}{dx^2}  \:dx^h \\[+1.5em]
		\text{Vecteur des efforts ext\'{e}rieurs répartis selon } \vec y \hspace{15pt}
		f_{i, py}^e&=&p_y^{e}  \dint_{x_1}^{x_2} \mathcal{N}_i  \:dx^h
	\end{array}
\tag{2.8}
\end{equation}$$
ou encore 
<a id="eq2.9"></a>
$$\begin{equation} 
\begin{array}{ccc}
	\mat{K^e_f} &=& \dint_e \mat{B_L^e}^T_f  \mat{\mathcal L_f}  \mat{B_L^e}_f  dx^h &&
	\vect{f^e_{py} }&= &\dint_e p_y <N(x^h)>^T  dx^h
\end{array}
\tag{2.9}
\end{equation}$$
Tous calculs faits, ces expressions ont la forme suivante dans le repère local de la poutre :
<a id="eq2.10"></a>
$$\begin{equation}
	\begin{array}{lr}
		\mat{K^e_f}= \dfrac{EI}{l_e^3}\begin{bmatrix}
			12 & 6l_e & -12 & 6l_e \\
			& 4l_e^2 & -6l_e & 2l_e^2 \\
			&  & 12 & -6l_e \\
			&  &  &  4l_e^2
		\end{bmatrix} & \ \ \
		\vect{ f^e_{py}} =
		\left(\dfrac{p_y\:l_e}{2}\right)
		\begin{Bmatrix}
			1  \\ \dfrac{l_e}{6} \\ 1 \\ -\dfrac{l_e}{6}
		\end{Bmatrix}
	\end{array}
\tag{2.10}
\end{equation} $$

## Rigidité en flexion-tension
Dans le repère local, ces contributions s'assemblent facilement selon les *ddl*s concernés. La matrice de rigidité en tension-flexion s'écrit dans la base locale, comme la somme des contributions des rigidités de flexion et de tension, en posant $\alpha = \dfrac{E^e\ S^e}{l_e}$ et $\beta =  \dfrac{E^e\ I^e}{l_e^3}$:
<a id="eq2.11"></a>
$$
\mat{K^e}_{(l)} = \left[ 
\begin{array}{cccccc|c}
     u_1 & v_1 & \theta_1 & u_2 & v_2 & \theta_2 &\\[0.2em]
    \hline
    \alpha & 0 & 0 & -\alpha & 0 & 0 & u_1 \\[0.5em]
     & 12 \beta & 6l_e \beta & 0 & -12 \beta & 6l_e \beta & v_1 \\[0.5em]
     &  & 4 l_e^{2} \beta & 0 & -6l_e \beta & 2l_e^2 \beta & \theta_1 \\[0.5em]
     &  &  & \alpha & 0 & 0 & u_2 \\[0.5em]
     & &  &  & 6l_e \beta & -6l_e \beta & v_2 \\[0.5em]
     &  &  & &  & 4l_e^2 \beta & \theta_2 \\[0.5em]
\end{array} \right]
\tag{2.11}
$$
Par contre, il faut que cette rigidité soit exprimée dans le repère de structure. Utilisons le changement de base introduit précédemment (<a href="main_EF_Hermite.ipynb#eq1.18">Eq. (1.18)</a>) : $\vect{q^e}_{(l)}=\mat{R}^T \vect{q^e}_{(g)}$, et reprenons l'origine de la matrice de rigidité qui traduit la discrétisation de la forme bilinéaire symétrique positive représentant l'énergie de déformation de la poutre
<a id="eq2.12"></a>
$$\begin{equation}
	\begin{array}{rccl}
	\vect{\delta q^e}_{(l)}^T \mat{K^e}_{(l)}\vect{q^e}_{(l)} = &\vect{\delta q^e}_{(g)}^T &\underbrace{\mat{R} \mat{K^e}_{(l)}  \mat{R}^T }&\vect{q^e}_{(g)} \\
	= & \vect{\delta q^e}_{(g)}^T &\mat{K^e}_{(g)} &\vect{q^e}_{(g)}
	\end{array}
\tag{2.12}
\end{equation}$$
ce qui conduit à l'expression finale :
<a id="eq2.13"></a>
$$
\mat{K^e}_{(g)} = \left[ 
\begin{array}{cccccc|c}
    u_1^1 & u_2^1 & \theta^1 & u_1^2 & u_2^2 & \theta^2 &\\
    \hline \\
    C^{2} \alpha + 12 S^{2} \beta & C S (\alpha - 12 \beta) & -6 S \beta l_e & -C^{2} \alpha - 12 S^{2} \beta & -C S (\alpha - 12 \beta) & -6 S \beta l_e & u_1^1 \\[0.5em]
     & 12 C^{2} \beta + S^{2} \alpha & 6 C \beta l_e & -C S (\alpha - 12 \beta) & -12 C^{2} \beta - S^{2} \alpha & 6 C \beta l_e & u_2^1 \\[0.5em]
     &  & 4 \beta l_e^{2} & 6 S \beta l_e & -6 C \beta l_e & 2 \beta l_e^{2} & \theta^1 \\[0.5em]
     &  &  & C^{2} \alpha + 12 S^{2} \beta & C S (\alpha - 12 \beta) & 6 S \beta l_e & u_1^2 \\[0.5em]
     &  &  &  & 12 C^{2} \beta + S^{2} \alpha & -6 C \beta l_e & u_2^2 \\[0.5em]
     &  &  &  &  & 4 \beta l_e^{2} & \theta^2 \\[0.5em]
\end{array} \right]
\tag{2.13}
$$

**Remarque** La projection dans le repère de structure peut aussi être réalisé au cas par cas à partir des des matrices de passage dans leur version numérique. Ici, comme les projections sont assez directes (dans le plan), on préfère utiliser ces formes analytiques.

## Assemblage des contributions élémentaires
Sans entrer dans les détails, les contributions élémentaires vont s'additioner pour former les grandeurs matricelles et vectorielles du système algébrique associé à l'équilibre. Les matrices resteront symétriques et définies positives dans notre cas. Cet __assemblage__ se fait sur la base de la somme des contributions associées aux noeuds de la discrétisation. En effet, la discrétisation permet de ramener le problème continu à une somme de contributions concentrées (efforts, raideurs, masses, amortissements, ...) aux noeuds du maillage.

Notons que les efforts extérieurs sont de 2 types, les efforts répartis qui sont exprimés aux noeuds $\vect{f}$, et les efforts ponctuels qui s'appliquent classiquement dans le repère de structure $\vect{F}$: 
<a id="eq2.14"></a>
$$
\vect{\mathcal{F}}_{(g)} = \vect{f}_{(g)} + \vect{F}_{(g)}
\tag{2.14}
$$

Pour le problème qu'on va devoir résoudre en statique, on repart de la forme intégrale faible discrétisée qui sera construite en sommant, pour chaque noeud $i$ du maillage, les contributions associées calculées pour chaque élément $e$. Pour les efforts, cette somme est assez directe - $\mathcal F_i = \sum \mathcal F_i^e$ -, et pour la raideur, on additionnera à chaque position $(i,j)$ les contributions $K^e_{ij}$ de tous les éléments s'appuyant sur ces noeuds $i$ et $j$.
<a id="eq2.15"></a>
$$
	< \ldots \ \delta q_i \ \ldots \ \ldots >_{(g)} \cdot 
	\left( 
  \left[ \begin{array}{cccc}
	 & & \vdots&  \\[+0.5em]
	\ldots &\ldots& K_{ij} = \dsum_{e=n}^k K^e_{ij}&\ldots\\[+0.5em]
	&&\vdots& \\[+0.5em]
	&&\vdots & \\
\end{array} \right]_{(g)}
\cdot
\begin{Bmatrix} \vdots \\ q_j \\ \vdots \\ \vdots \end{Bmatrix}_{(g)}
=\begin{Bmatrix} \vdots \\ \mathcal F_i \\ \vdots \\ \vdots \end{Bmatrix}_{(g)}
\right) =0, \ \ \forall \vect{\delta q}^T \ CA^h(0)
\tag{2.15}
$$

L'assemblage s'appuie sur les **connectivités** des éléments, *i.e.* les noeuds qu'ils partagent. Le système linéaire à résoudre correspond en fait à l'équilibre des efforts intérieurs et des efforts extérieurs, c'est en fait le **résidu** noté $\vect{\Psi}$ par exemple. Le vecteur déplacement solution s'obtient (souvent) par *inversion* de la rigidité
<a id="eq2.16"></a>
$$
\boxed{\vect{\Psi(\vect q^e)}=\mat{K}_{(g)} \cdot \vect{q}_{(g)}- \vect{\mathcal{F}}_{(g)} =0 \Rightarrow \vect{q}_{(g)} = \mat{K}^{-1}_{(g)} \cdot \vect{\mathcal{F}}_{(g)} }
\tag{2.16}
$$

**Remarque :** Ce sytème à inverser est singulier tant que les conditions aux limites de Dirichlet n'ont pas été imposées. Cela correspond physiquement à l'impossibilité de générer des déformations dans les solides tant que les mouvements de corps rigide ne sont pas bloqués. En termes numériques, cela correspond par exemple à un ou des pivots nuls dans le processus de décomposition de Cholesky très souvent utilisé pour inverser la rigidité.

## Généralisation
En introduisant la relation générique entre les déplacements et les déformations (<a href="main_EF_Hermite.ipynb#eq1.15">Eq. (1.15)</a>), on peut formuler cet équilibre de façon pour tout solide (en HPP).On introduit les efforts conjugés aux déformations, ici les conposantes du second tenseur de Piola-Kirchhoff, qui s'écrivent sous la forme d'un vecteur des contraintes $\vect{\mathcal{S}^e(\vect{\gamma^e})}$ de la même dimension que les déformations de façon à ce que l'énergie de déformation s'écrive $\vect{\mathcal{S}^e(\vect{\gamma^e})}^T \cdot \vect{\gamma^e(\vect{q^e})} $. La forme intégrale faible élémentaire s'écrit alors :
<a id="eq2.17"></a>
$$  -\dint_{\Omega^{0}_{e}} \vect{\mathcal{S}^e(\vect{\epsilon^e})}^T \mat{B^e_L} \vect{\delta q^e} d\Omega^{0}_e + \vect{\delta q^e}^T\vect{\mathcal F^e}
\tag{2.17}
$$
et l'équilibre global s'écrit, après assemblage:
<a id="eq2.18"></a>
$$ \begin{array}{rclcl}
% -\dint_{\Omega^{0}_{e}} \vect{\mathcal{S}^e(\vect{\gamma^e})}^T \mat{B^e_L} \vect{\delta q^e} d\Omega^{0}_e %\\[+1em]
%+ \vect{\delta q^e}^T\left(  \vect{f(\vec{x})^e} + \vect{F^d(\vec{x})^e} \right) \\
		 & & \vect{\delta q} ^T \left( \dint_{\Omega^0} \mat{B_L}^T \vect{\mathcal{S}(\vect{\epsilon})}d\Omega^0 \right.& - &
	\left.	\vect{\mathcal{F}} \vphantom{\ \dint_{\Omega^0}}  \right)=  0, \ \forall \vect{\delta q}^T \ C.A.^h(0)  \\[+1em]
	&=& \vect{\delta q}^T \left( \underbrace{\dint_{\Omega^0} \mat{B^e_L}^T  \mat{\mathcal L} \mat{B^e_L} d \Omega^0 } \vect{q} \right. &-& \left.	\vect{\mathcal{F})} \vphantom{\ \dint_{\Omega^0}}  \right) \\[+2.5em]
	\Rightarrow&\vect{\Psi(\vect{q})}  \equiv  &  \hspace{7.5em} \mat{K}_{(g)} \hspace{2em} \cdot \hspace{2em} \vect{q}_{(g)} &-& \vect{\mathcal{F}}_{(g)} =  \vect{0} \ ,  \forall \vect{\delta q}^T \ C.A.^h(0)
	\end{array}
\tag{2.18}
$$
    
## Prise en compte des conditions de Neumann et de Dirichlet
**Neumann**
Nous avons vu ci-dessus que les efforts extérieurs s'écrivent comme la somme des efforts répartis et des efforts ponctuels : $
\vect{\mathcal{F}}_{(g)} = \vect{f}_{(g)} + \vect{F}_{(g)}$. Les efforts ponctuels, nos conditions de Neumann, seront appliqués directement aux *ddl*s correspondants dans $\vect F$, et la contribution des efforts répartis sera d'abord évaluée dans le repère locale, puis projetée explicitement dans le repère de structure. Cette projection se fait en repartant en repartant de leur contribution dans la puissance virtuelle associée 
<a id="eq2.19"></a>
$$\begin{equation}
	\begin{array}{l}
\vect{\delta q^e}_{(g)}^T\vect{f^e}_{(g)} = \vect{\delta q^e}_{(l)}^T \vect{f^e}_{(l)}=\left(\mat{R}^T \vect{\delta q^e}_{(g)}\right)^T\vect{f^e}_{(l)} = \vect{\delta q^e}_{(g)}^T\mat{R}\vect{f^e}_{(l)}  \\[+3em]
 \leadsto  \vect{f^e}_{(g)}  = \left(\dfrac{l_e}{2}\right)  \begin{Bmatrix} C \ p_x - S \ p_y\\ S\ p_x  + C \ p_y\\ p_y \dfrac{l_e}{6} \\ C \ p_x - S \ p_y\\ S\ p_x  + C \ p_y \\ -p_y \dfrac{l_e}{6} \end{Bmatrix}
\end{array}
\tag{2.19}
\end{equation}$$

**Dirichlet homogènes**
Les conditions de Dirichlet homogènes sont prises en compte en mettant à l'unité les contributions diagonales dans les matrices de rigidité et en annulant les autres contributions liées à ce *ddl* *i*, de sorte que $K_{ii} u_i = 0 \Rightarrow u_i=0$:
<a id="eq2.20"></a>
$$
q_i=0 \Rightarrow \begin{cases} K_{ii}=1 \\ K_{ij}=0 \\ \mathcal{F}_i=0\end{cases}
\tag{2.20}
$$

**Dirichlet non-homogènes**
Pour les conditions non-homogènes, il existe différentes méthodes pour les imposer - *cf* https://www.emse.fr/~drapier/index_fichiers/CoursPDF/Meca-Structu2A/SDrapier_Meca-struct-num_2024.pdf, Chapitre 6- , dont la méthode par pénalité que nous décrivons ici. Le principe reste cependant le même : le système correspondant à ces conditions s'écrit en faisant apparaître les **réactions** $\vect{R_b}$ qui ne sont habituellement pas présentes dans le système à résoudre et qui correpondent aux $n_D$ *ddl*s imposés notés $\vect{q_b}$; les $n-n_D$ *ddl*s libres étant notés $\vect{q_l}$. En effet, dans le cas général où $\vect{\delta q}$ n'est pas CA$^h$(0) mais simplement CA$^h$, le travail virtuel produit par ces efforts extérieurs: $\int_{\partial \Omega_{u}} \tens{\sigma} \cdot \vec{n} \cdot \delta \vec{u} d\Omega \neq 0$ en 3D. Le PPV s'écrit alors 
<a id="eq2.21"></a>
$$ 	\vecd{\vect{\delta q_l}} { \vect{\delta q_b}}^{T}   
	\left[ \begin{array}{c|c}
		\mat{K_{ll}}&\mat{K_{bl}}\\[+0.5em]
		\hline \\[-1em]
		\mat{K_{lb}} & \mat{K_{bb}} 
	\end{array}\right] \cdot
	\vecd{\vect{q_l}} { \vect{q_b}}  \\[.8em] 
	- \vecd{\vect{\delta q_l}} { \vect{\delta q_b}} ^T \vecd{\vect{\mathcal{F}_l}} {\vect{\mathcal{F}_b}}  - \vect{\delta q_b }^T \vect{R_b} =0  \;, \forall \vecd{\vect{\delta q_l}} { \vect{\delta q_b}}^T C.A.^h
\tag{2.21}
$$

$\mat{K_{ll}}$ et $\mat{K_{bb}}$ sont les sous-matrices carrées correspondant aux *ddl*s bloqués et libres respectivement, et des vecteurs d'efforts connus $\vect{\mathcal{F}_l}$ et $\vect{\mathcal{F}_b}$ y sont associés. On notera que ce dernier vecteur ne peut en aucun cas correspondre à des efforts ponctuels, puisqu'on ne peut à la fois imposer une condition en déplacement et en effort simultanément sur le même *ddl*. Ces efforts correspondent donc aux contributions des efforts répartis rapportées aux noeuds, soit $\vect{\mathcal{F}_b} \equiv \vect{f}$.

Les sous-matrices rectangulaires $\mat{K_{bl}}$ et $\mat{K_{lb}}$ relient les contributions croisées des *ddl*s imposés et inconnus. Le déplacement recherché $\vect{q_l}$ sous la condition $\vect{q_b}=\vect{q^d}$ est donc solution de :
<a id="eq2.22"></a>
$$\mat{K_{ll}}\vect{q_l}= \vect{\mathcal{F}_l}-\mat{K_{bl}}^T\vect{q_b}
\tag{2.22}
$$

On voit que cette méthode nécessite de traiter un sous-problème, on peut procéder de façon plus directe. On va chercher à vérifier $\vect{q_b} = \vect{q^d}$ en imposant une réaction 'numérique' supplémentaire qui traduit la réponse du milieu à cette condition cinématique. Cette réaction s'écrit $\vect{R_b}=\alpha\left(\vect{q_b} - \vect{q^d} \right)$ avec $\alpha$ un scalaire - une **pénalité** - à choisir grand (de l'ordre de $10^{5-8}$, ou plus généralement $\gg max(K_{ij})$). Le travail virtuel des efforts de réaction devant être nul, la condition sera d'autant mieux vérifiée que $\alpha$ sera grand - $\left(\vect{q_b} - \vect{q^d} \right) \searrow 0$ pour conserver $\vect{R_b}$ du même ordre de grandeur si $\alpha \nearrow$.
<a id="eq2.23"></a>
$$\begin{equation}
  \begin{array}{l}
\left[ \begin{array}{c|c}
     \mat{K_{ll}}&\mat{K_{bl}}^{T}\\ 
     \hline \\
      \mat{K_{bl}} & \mat{K_{bb}}+\alpha \mat{I}
 \end{array}\right]
\begin{Bmatrix} \vect{q_l} \\ \vect{q_b} \end{Bmatrix}
 =\begin{Bmatrix} \vect{\mathcal{F}_l}  \\ \vect{f_b}+ \alpha \vect{q^d} \end{Bmatrix}
\end{array}
\tag{2.23}
\end{equation}$$
Cette méthode est également connue sous la dénomination de la *méthode du terme diagonal dominant*, d'après la modification $\mat{K_{bb}} \leadsto \mat{K_{bb}}+\alpha \mat{I} $. Elle est de mise en oeuvre plus simple que la méthode précédente (élimination) car il suffit d'ajouter la pénalité sur le terme diagonal de la rigidité et sur le second membre du *ddl* concerné, sans modifier la structure du problème après assemblage. Comme dans tout problème de minimisation sous contraintes, d'autres méthodes existent où la contrainte est imposée dans la forme intégrale faible : la méthode des *multiplicateurs de Lagrange*, la méthode du *Lagrangien augmenté*, ou encore la *formulation mixte intégrant les réactions* comme inconnues.

**Calcul des réactions** Le calcul des réactions *a posteriori* se fait directement quand le stockage des *ddl*s est prévu comme indiqué ci-dessus, ce qui nécessite de passer par une numérotation interne adaptée. Une solution moins contraignante, bien qu'approximative, consiste à utiliser les réactions calculées dans les éléments qui sont liés à une condition de Dirichlet, quitte à réaliser des moyennes des efforts des éléments s'appuyant sur ce *ddl*. On peut aussi considérer tous les *ddl*s du problème et obtenir les réactions pour un coût numérique réduit via la relation ci-dessus réécrite puisque maintenant tous les déplacements sont déterminés : 
<a id="eq2.24"></a>
$$ \vecd{\vect{0}} {\vect{R_b}}  = \left[ \begin{array}{c|c}
		\mat{K_{ll}}&\mat{K_{bl}}\\
		\hline \\[-1em]
		\mat{K_{lb}} & \mat{K_{bb}} 
	\end{array}\right] \cdot
	\vecd{\vect{q_l}} { \vect{q_b}} - \vecd{\vect{\mathcal{F}_l}} {\vect{f_b}}
\tag{2.24}
$$   
**Précaution** L'utilisation de cette forme complète de l'équilibre discrétisé, *i.e.* ne se limitant pas aux *ddl*s bloqués, nécessite de revenir à la forme intégrale faible initiale caractérisant l'équilibre, *i.e.* non contrainte par les conditions de Dirichlet. De plus, si les conditions de Dirichlet, homogènes ou non-homogènes sont imposées par pénalité, il faut **veiller à retrancher les contributions associées dans la rigidité et le second membre**, pour revenir à la forme intégrale faible initiale caractérisant l'équilibre, *i.e.* non contrainte par les conditions de Dirichlet non-homogènes.
Une solution sûre peut consister à ne modifier que temporairement les grandeurs algébriques pour la résolution du problème statique puis imposer à nouveau les CL homogènes, ou bien de conserver une copie du système construit avant la prise en compte des CL de Dirichlet.

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Novembre 2024

@author: S. Drapier

"""

import sys
path = "exemples/"
#path = "../source/Python_Files/"
#path = ""
sys.path.append(path)

import numpy as np
# recharge automatiquement les modules (Hermite_NL_Classes, affichage_interactif) s'ils changent sur le disque
%load_ext autoreload
%autoreload 2

import Hermite_NL_Classes as Cl
import affichage_interactif as AI  # affichages interactifs (Plotly + ipywidgets)


# Initialisation de ce qui est de dimension connue
nb_elem = 0
nb_nodes = 0
    
nodes_global = []
fixed_dofs = []
elem_global = []
nodes_elem = []

In [15]:
""" 
    1/ Maillage = 
- définitions des noeuds = construire la liste nodes_global avec des objets noeuds; 
peut être fait à partir d'un fichier texte contenant 3 colonnes : num_noeud, x, y'
- définition des connectivités = donner une liste nodes_elem;
peut être fait à partir d'un fichier texte contenant 3 colonnes : num_elem, noeud1, noeud 2
l'ordre des noeuds définit l'orientation : les charges réparties sont appliquées dans le repère LOCAL

Utilisation d'une fonction pour 1 poutre simple (tests) ou lecture depuis fichiers de définition des noeuds et des connectivités
"""

# A/ maillage d'une poutre test : définition du maillage de 'Nl' éléments orientés d'un angle 'angle'
# Fonction de Maillage d'une poutre droite : Cl.fonct_Maille_Poutre(longueur, Nl, angle)  (angle en DEGRES)
longueur = 1.
Nl = 20
angle = 15.  # en degrés, par rapport à x1

""" Création de la liste des noeuds - la position du noeud en partant de 0 est celle qui servira
à stocker les ddls, indépendamment de son numéro """
#nodes_global, nodes_elem = Cl.fonct_Maille_Poutre(longueur, Nl, angle)

# B/ Maillage à partir de 2 fichiers: définition des noeuds (num_node, x, y) et défintion des éléments (num_elem, inode1, inode2) 
nodes_global, nodes_elem = Cl.fonct_Maille_depuis_Fichier(path+'pont_noeuds.txt', path+'pont_elem.txt')
nb_nodes = len(nodes_global)
nb_elem = len(nodes_elem)

# Les propriétés et chargements répartis des éléments peuvent être précisés dans un np.array
E = 70.E9
rho = 2.6E3
bs = 0.15
hs = 0.3
ES = E * bs * hs
EI = E * bs * hs**3/12
#EI = 1
Smasse = rho * bs * hs # masse de la section
meca_elem = np.repeat([[ES, EI,Smasse]],nb_elem,axis=0)

px = 0. #9.81 * rho * bs * hs
py = 0.
px_py = np.repeat([[px, py]],nb_elem,axis=0)

props_elem = [meca_elem, px_py]

In [16]:
""" 
    2/ Créé un objet "problème éléments finis" à partir d'une liste de noeuds et des propriétés Meca_elem et efforts linéiques px_py
"""
pb = Cl.pbEF_poutre(nodes_global, nodes_elem, props_elem)

# Créé le maillage correspondant aux connectivités et assemble les grandeurs (K_global, M_global, F_rep)
pb.Maillage_Assemblage()

""" 
    3/ Conditions aux limites
"""
total_dofs = pb.total_dofs

pb.CL_forces = [[11,2,-2.5E3]]  # effort transverse 3 N, angle en degrés #[[nb_nodes,2,3]]#[[11,2,-5E3]]#[,1,-1/np.sqrt(2)],[11,2,-1./np.sqrt(2)]]#,[3,1,2E3]]
pb.CL_deplacements = [[1,1,0],[1,2,0],[1,3,0],[11,3,0],[11,1,0]]#,[nb_nodes,1,0],[nb_nodes,3,0],[nb_nodes,2,-1E-2]]

pb.Application_CL()


K_global, Kgeom_global, M_global = pb.K_global, pb.Kgeom_global, pb.M_global
F_con, F_rep, F_tot = pb.F_con, pb.F_rep, pb.F_tot

In [ ]:
#Cl.Creation_Elem_Assemblage(pb, 20, 5,9, [ES, EI,Smasse], [px,py])

array([[1.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       [0.000e+00, 1.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       [0.000e+00, 0.000e+00, 1.000e+00, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       ...,
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 1.000e+00, 0.000e+00,
        0.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 2.268e+09,
        0.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
        1.000e+00]])

In [17]:
"""
    4/ Résolution 
"""    
dofs = pb.Resol()

# Mise à jour des champs cinématiques et statiques
elem_global = pb.Calcul_Def_Eff()

""" 
    5/ Affichage interactif : champ (u, N, T, M, K...), couleur sur la déformée ou diagramme, amplification,
    valeurs au survol de la souris (efforts internes dans les éléments, ddls et réactions aux noeuds)
""" 
AI.vue(pb, champ='u')
# Version statique (matplotlib), pour impression ou export PDF :
# pb.Affichage('u', 'coolwarm', vp = 5)   # sans colormap (coolwarm, bwr, seismic...), pas de colorbar


In [11]:
# Éléments : géométrie, charges réparties (repère local), efforts internes N, T, M
AI.tableau_elements(pb).show()
pb.dofs
# print(elem_global[1])   # version texte, pour un élément

array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  5.29100529e-05,
       -7.93650794e-07, -2.11640212e-04,  2.11640212e-04, -1.58730159e-06,
       -4.23280423e-04,  4.76190476e-04, -2.38095238e-06, -6.34920635e-04,
        8.46560847e-04, -3.17460317e-06, -8.46560847e-04,  1.32275132e-03,
       -3.96825397e-06, -1.05820106e-03,  1.90476190e-03, -4.76190476e-06,
       -1.26984127e-03,  1.90476190e-03, -6.88183422e-04, -1.45502646e-03,
        1.90476190e-03, -1.45097002e-03, -1.58730159e-03,  1.90476190e-03,
       -2.26666667e-03, -1.66666667e-03,  1.90476190e-03, -3.10881834e-03,
       -1.69312169e-03])

In [6]:
""" Calcul des réactions : se fait sur le système avant prise en compte des CL de Dirichlet: on travaille sur une copie
    R_b = K_global_copie * dofs - F_tot_copie """

reactions = pb.Calcul_Reactions()
# Noeuds : déplacements (u1, u2, θ) et réactions (R1, R2, Mr), dans le repère de structure
AI.tableau_noeuds(pb).show()
# Version texte :
# print(pb.reactions)
# print(pb.nodes_global[6])
# print(pb.nodes_global[10])

In [7]:
# %%
# Déformée à l'échelle réelle, comparée à la solution analytique de la console des cellules 2-3 :
# encastrée en x = 0, charge transverse répartie py (repère local, px = 0) et effort transverse P en bout
#   v(x) = py x² (6L² - 4Lx + x²) / 24EI  +  P x² (3L - x) / 6EI        (u = 0 : pas de charge axiale)
# /!\ à adapter si le maillage, les charges ou les CL des cellules 2-3 changent
P = 3.  # effort transverse en bout, cf. pb.CL_forces
C, S = np.cos(np.radians(angle)), np.sin(np.radians(angle))
x = np.linspace(0, longueur, 101)
v = py*x**2*(6*longueur**2 - 4*longueur*x + x**2)/(24*EI) + P*x**2*(3*longueur - x)/(6*EI)
fig = AI.figure(pb, 'u', ampli=1., titre='Déformée (échelle réelle) : EF et solution analytique')
fig.add_scatter(x=x*C - v*S, y=x*S + v*C, mode='lines', name='analytique', line=dict(dash='dash', color='black'))
fig.show()
n_bout = pb.nodes_global[-1]
print(f"Flèche en bout : EF = {-n_bout.u1*S + n_bout.u2*C:.6e}   analytique = {v[-1]:.6e}")

# Version statique (matplotlib) :
# # # Tracé de la déformée, par exemple pour comparaison avec solution analytique
# X = np.zeros(Nl+1)
# Y = np.zeros(Nl+1)
# # x1_values = np.zeros(Nl+1)
# # x2_values = np.zeros(Nl+1)
# x1_values = [node.x1 for node in nodes_global]
# x2_values = [node.x2 for node in nodes_global]
# for ind in range(Nl+1):
#     X[ind] = nodes_global[ind].x1 + dofs[ind].u1
#     Y[ind] = nodes_global[ind].x2 + dofs[ind].u2
#
# import matplotlib.pyplot as plt    
# plt.plot(x1_values,x2_values,'x-', color='b',label='initial')
# plt.plot(X,Y,'o-', color='r',label='deformee')
# plt.legend()
# plt.title('A changer')  
# plt.show()


Flèche en bout : EF = 2.000000e+00   analytique = 2.000000e+00


In [ ]:
#%% Barre sous son propre poids + déplacement imposé - Chapitre 6
E = 70.E9
l = 2
bs = 0.15
hs = 0.3
rho = 2.6E3
g = 9.81

E = 70.E9
rho = 2.6E3
bs = 0.15
hs = 0.3
ES = E * bs * hs
EI = E * bs * hs**3/12
Smasse = rho * bs * hs # 

EI=E*bs*hs**3/12
ES = E*bs*hs

px = g*rho*bs*hs

u_d = -0.1

def u_l(x):
    return u_d*x/l - (px/2*E)*x*(x-l)

def N(x):
    return ES*u_d/l - (px/2)*(2*x-l)

print(f"u(l)= {u_l(l):.6e} \n Rd = {N(l):.6e}")

In [8]:
#%% Portique: mât de hauteur hp et potence de longueur lp

E=70.E9
lp = 2
hp = 3
bs = 0.15
hs = 0.3
F=5.E3

EI=E*bs*hs**3/12
ES = E*bs*hs
F_sur_EI = F/EI

v_A = F_sur_EI*lp*hp**2/2 *1000
phi_A = -F_sur_EI*lp*hp 
v_B = -F_sur_EI*lp**2*(hp+lp/3) *1000 #- (F/ES)*h*1000
phi_B = -F_sur_EI*lp*(hp+lp/2)

print(f"u_A= {v_A} mm \n phi_A= {phi_A} rad \n v_B= {v_B} mm \n phi_B= {phi_B} rad ")

u_A= 1.904761904761905 mm 
 phi_A= -0.00126984126984127 rad 
 v_B= -3.104056437389771 mm 
 phi_B= -0.0016931216931216934 rad 


Exemple de sortie de `print(pb.nodes_global[6])` et `print(pb.nodes_global[10])` pour le portique :

```text
Noeud 7, x1 = 0.000000, x2 = 3.000000, 
 u = 0.000044, v = -0.000182, 
 u1 = 0.000044, u2 = -0.000182, 
 theta = -0.004426917317037545 
 position = 6
Noeud 11, x1 = 2.000000, x2 = 3.000000, 
 u = 0.000000, v = -0.010000, 
 u1 = 0.000000, u2 = -0.010000, 
 theta = 0.0 
 position = 10
```